# 9장 · 인과 주장은 설계에 기댄다

이 노트북은 구글 **Colab**에서 바로 실행됩니다. 설치는 없고, 구글 계정만 있으면 됩니다.

**읽는 법.** 지금 읽고 있는 이런 글은 설명이라 실행할 것이 없습니다. 코드는 **왼쪽에 `[ ]` 와 ▶ 단추가 붙은 상자**이고, 그 ▶ 또는 `Shift`+`Enter` 로 실행합니다.

**순서.** 번호 차례대로 끝까지 갑니다. **1 준비**부터 시작해 마지막 번호까지 위에서 아래로 내려가면 됩니다. ⚠ 가운데부터 누르면 앞에서 만든 것이 없어 오류가 납니다.

📖 본문 학습 페이지: [9장 · 인과 주장은 설계에 기댄다](https://grow.minds.kr/textbooks/css-methods/causal/book/ch09-설계가-인과를-산다.html)

## 1. 준비

아래 상자 **둘**을 차례로 실행하세요. 둘 다 해야 그다음이 돌아갑니다.

1. 첫째 = 자료와 코드를 내려받습니다. 몇 초 걸리고 「경고」 문구가 떠도 정상입니다. 상자 아래에 `/content/css-methods-causal-code` 가 찍히면 성공입니다.
2. 둘째 = 도구를 불러옵니다. **준비 끝**이라는 줄이 찍히면 됩니다.

⚠ 한참 뒤에 돌아와 `pd`·`exp`가 없다거나 파일이 없다는 오류가 나면 코랩이 연결을 끊은 것입니다. 자료는 내 컴퓨터가 아니라 코랩의 컴퓨터에 있었으므로 함께 사라졌습니다. **이 두 상자를 위에서부터 다시** 누르면 됩니다(첫째 상자는 여러 번 눌러도 됩니다).

> **어느 것이 상자인가.** 왼쪽에 `[ ]` 표시와 **▶** 단추가 붙은 것이 코드 상자입니다. 글씨에 색이 섞여 있습니다. 지금 읽고 있는 이런 글에는 `[ ]` 도 ▶ 도 없습니다.

In [ ]:
# 이 책의 데이터·코드를 코랩으로 내려받습니다(수 초). 다시 눌러도 됩니다.
%cd /content
![ -d css-methods-causal-code ] || git clone -q https://github.com/dataminds/css-methods-causal-code.git
%cd /content/css-methods-causal-code

In [ ]:
import pandas as pd, numpy as np
from scipy import stats

def load(name, clean=True):
    df = pd.read_csv(f"data/journey_{name}.csv")
    return df[df.attn_1 == 1] if clean and "attn_1" in df else df

def ols(y, X):                      # 절편 포함 최소제곱 → (계수, 표준오차, p, R^2)
    y = np.asarray(y, float)
    X1 = np.column_stack([np.ones(len(y))] + [np.asarray(x, float) for x in X])
    b, *_ = np.linalg.lstsq(X1, y, rcond=None)
    resid = y - X1 @ b
    n, k = X1.shape
    se = np.sqrt(np.diag(resid @ resid / (n - k) * np.linalg.inv(X1.T @ X1)))
    p = 2 * stats.t.sf(np.abs(b / se), n - k)
    r2 = 1 - (resid @ resid) / ((y - y.mean()) @ (y - y.mean()))
    return b, se, p, r2

def cohen_d(a, b):
    sp = np.sqrt(((len(a)-1)*a.std(ddof=1)**2 + (len(b)-1)*b.std(ddof=1)**2) / (len(a)+len(b)-2))
    return (a.mean() - b.mean()) / sp

def cronbach(items):
    items = np.asarray(items, float); k = items.shape[1]
    return k/(k-1) * (1 - items.var(axis=0, ddof=1).sum() / items.sum(axis=1).var(ddof=1))

print("준비 끝. 데이터와 도우미 함수를 불러왔습니다.")


💡 **방금 둘째 상자에 코드가 길게 펼쳐졌는데, 지금 이해하지 않아도 됩니다.**

이 책 전체가 쓰는 **도우미 넷**을 미리 만들어 두는 곳이라 그렇습니다. 지금은 이름과 하는 일만 훑고 넘어가세요. 안에 든 식은 필요한 장에서 하나씩 만납니다.

| 이름 | 하는 일 | 처음 쓰는 곳 |
|---|---|---|
| `load` | 자료 한 벌을 불러오고 주의 점검 실패자를 걸러 낸다 | 2장 |
| `cronbach` | 척도 신뢰도(알파) | 4장 |
| `cohen_d` | 두 집단 차이의 효과크기 | 13장 |
| `ols` | 회귀의 계수·표준오차·p·R² | 14장 |

⚠ `np.linalg.lstsq` 처럼 낯선 이름이 보여도 괜찮습니다. **그것이 이 책이 「계산은 AI가」라고 말하는 곳입니다.**

⭐ **그리고 이 넷은 여러분이 검증할 대상이 아닙니다.** 이 책이 검증해 둔 도구입니다(배포 때마다 대조 배터리가 이 함수들의 출력을 본문 수치와 맞춰 봅니다). 여러분이 읽어야 할 코드는 **AI가 준 코드**이고, 그 훈련은 3장부터 시작합니다.

📖 이 넷을 다시 보고 싶으면 책 부록 E.1b 에 같은 코드가 있습니다.

## 2. 무작위 대 자기선택 (실습 1)
400명을 동전으로 나눈 경우와 낙관적인 사람이 스스로 고른 경우를 견줍니다. 네 변수(나이 · 성별 · 낙관성 · 끈기)의 집단 차이 *d*를 찍습니다. 낙관성과 끈기는 실제 연구라면 측정하지 못한 변수입니다.

In [ ]:
rng = np.random.default_rng(73)
n = 400
age  = rng.normal(35, 10, n); sex  = rng.integers(0, 2, n)   # 우리가 측정한 것
opt  = rng.normal(0, 1, n);   grit = rng.normal(0, 1, n)     # 우리가 측정하지 못한 것

z  = rng.integers(0, 2, n)                                    # (가) 동전이 정한다
p  = 1 / (1 + np.exp(-1.2 * opt))                             # (나) 낙관적인 사람이 스스로 고른다
zs = (rng.random(n) < p).astype(int)

for lab, g in [("무작위", z), ("자기선택", zs)]:
    print(lab, [float(round(cohen_d(v[g == 1], v[g == 0]), 2)) for v in (age, sex, opt, grit)])
# 무작위  [-0.02, -0.08, -0.05, -0.11]
# 자기선택 [-0.18, 0.04, 1.03, -0.12]

## 3. 단순 무작위 대 블록 무작위 (실습 2)
같은 400명을 2,000번 새로 나눕니다. 판마다 두 집단의 기저 차이(*d*)를 모아 평균(편향) · 표준편차(변동) · |*d*| > 0.2 비율을 찍습니다. ⚠ 바로 위 셀의 `rng`와 `n`을 이어 쓰므로 위 셀을 먼저 실행하세요. 시간이 조금 걸립니다.

In [ ]:
base = rng.normal(0, 1, n)                    # 기저 의미감 (참값)
pre  = base + rng.normal(0, 0.5, n)           # 사전검사 = 그 대리 지표
order = np.argsort(pre)
simple, block = [], []
for _ in range(2000):
    z = rng.integers(0, 2, n)
    simple.append(cohen_d(base[z == 1], base[z == 0]))
    g = np.zeros(n, int)
    for i in range(0, n, 2):                  # 사전검사가 비슷한 둘씩 짝지어 하나만 처치
        g[order[i:i + 2][rng.integers(0, 2)]] = 1
    block.append(cohen_d(base[g == 1], base[g == 0]))
for lab, v in [("단순", simple), ("블록", block)]:
    v = np.array(v)
    print(lab, round(v.mean(), 3), round(v.std(ddof=1), 3), round(np.mean(np.abs(v) > 0.2), 4))
# 단순 0.001 0.098 0.0345
# 블록 0.0 0.048 0.0

## 4. 패널: 개인 안에서 보면 관계가 줄어든다
횡단 상관 .29가 개인 평균 중심화 후 .06으로. 대부분이 안정 성향의 몫이었습니다.

In [ ]:
panel = pd.read_csv("data/journey_panel.csv")
w1 = panel[panel.wave==1]
within = panel[["hjs","mil"]] - panel.groupby("id")[["hjs","mil"]].transform("mean")
print("횡단 r:", round(w1.hjs.corr(w1.mil),2), " 개인 내 r:", round(within.hjs.corr(within.mil),2))

## 5. 같은 상관을 내는 세 세계 (실습 4)
A(여정→다음 의미) · B(의미→다음 여정) · C(안정된 성향만, 서로 영향 없음). 횡단 상관은 셋 다 0.2대이고, 시차 기울기는 A와 B를 가르지만 C에서는 영향 없이 두 방향이 다 나옵니다.

In [ ]:
def 시차_세계(seed=73, n=5000):
    g = np.random.default_rng(seed)
    e = lambda s=1: g.normal(0, s, n)
    x1 = e(); y1 = .3*x1 + e(); x2 = .6*x1 + e(.8); y2 = .5*y1 + .3*x1 + e(.8)
    A = (x1, y1, x2, y2)
    y1 = e(); x1 = .3*y1 + e(); y2 = .6*y1 + e(.8); x2 = .5*x1 + .3*y1 + e(.8)
    B = (x1, y1, x2, y2)
    T = e()
    C = tuple(.64*T + e() for _ in range(4))
    return {"A 여정→의미": A, "B 의미→여정": B, "C 성향만": C}

def 시차(앞_자기, 원인, 뒤):                       # 앞 시점 자기 값이 비슷한 사람끼리 구한 기울기
    q = pd.qcut(앞_자기, 10, labels=False)          # 앞 시점 자기 값을 열 층으로
    return float(np.mean([np.polyfit(원인[q == k], 뒤[q == k], 1)[0] for k in range(10)]))

for name, (x1, y1, x2, y2) in 시차_세계().items():
    print(name, round(np.corrcoef(x1, y1)[0, 1], 2),
          round(시차(y1, x1, y2), 2), round(시차(x1, y1, x2), 2))
# A 0.23 0.31 -0.0 / B 0.28 0.01 0.31 / C 0.29 0.23 0.22

## 6. 직접 바꿔 보기
위 셀의 숫자(씨앗 73, 표본 크기, 제외 기준 등)를 바꿔 다시 실행해 보세요. 결과가 어떻게 달라지나요?

> **검증 로그(부록 B)**: 무엇을 바꿨고, 무엇이 나왔고, 예상과 같았는지 한 문단으로 적어 두세요. 실행이 아니라 검증이 이 책의 핵심입니다.